# Training Curves for MNIST

Track loss and accuracy over time so students can see whether a model is learning, stalling, or overfitting.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)

def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

device = get_device()

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_data = datasets.MNIST(root="../../data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="../../data", train=False, download=True, transform=transform)
train_loader = DataLoader(Subset(train_data, range(10000)), batch_size=128, shuffle=True)
test_loader = DataLoader(Subset(test_data, range(2000)), batch_size=256)

In [ ]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += loss_fn(logits, labels).item()
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

history = {"train_loss": [], "test_loss": [], "test_accuracy": []}

for epoch in range(8):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(model(images), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    test_loss, test_accuracy = evaluate(model, test_loader)
    history["train_loss"].append(train_loss / len(train_loader))
    history["test_loss"].append(test_loss)
    history["test_accuracy"].append(test_accuracy)
    print(f"epoch {epoch + 1}: train_loss={history['train_loss'][-1]:.4f}, test_loss={test_loss:.4f}, test_accuracy={test_accuracy:.3f}")

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(epochs, history["train_loss"], label="train loss")
axes[0].plot(epochs, history["test_loss"], label="test loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross entropy")
axes[0].legend()

axes[1].plot(epochs, history["test_accuracy"], label="test accuracy", color="tab:green")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].set_ylim(0, 1)

plt.tight_layout()

## Discussion

A falling training loss means the optimizer is finding parameters that fit the training set. A widening gap between training and test loss is a first sign of overfitting.